In [1]:
import torch
import os
%set_env TOKENIZERS_PARALLELISM=false
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

env: TOKENIZERS_PARALLELISM=false
Using device: cuda


In [2]:
from transformer_lens import HookedESM3,SupportedESM3Config
from esm.pretrained import (
    ESM3_sm_open_v0,
)
from esm.models.esm3 import ESM3
import random
import torch.nn.functional as F
from esm.tokenization import get_esm3_model_tokenizers


In [3]:
from random import randint
from esm.tokenization import get_esm3_model_tokenizers

def prepare_sequences(proteins_list, device):
    tokenizers = get_esm3_model_tokenizers()
    labels = []  # To store true labels (masked characters)
    masked_indices = []  # To store positions of masked tokens
    masked_proteins = []  # To store masked protein sequences

    # Randomly mask one token per protein
    for protein in proteins_list:
        mask_position = randint(0, len(protein) - 1)
        true_label = protein[mask_position]  # Get the true label (original character)
        labels.append(true_label)
        masked_indices.append(mask_position)

        # Replace the selected position with a mask token
        masked_protein = protein[:mask_position] + tokenizers.sequence.mask_token + protein[mask_position + 1:]
        masked_proteins.append(masked_protein)
    masked_indices = torch.tensor(masked_indices)
    masked_indices+=1 #padding with bos
    # Tokenize the masked sequences
    tokenizers_result = tokenizers.sequence(
        masked_proteins,
        return_tensors="pt",
        add_special_tokens=True,
        padding=True
    )
    input_ids = tokenizers_result['input_ids']
    sequence_ids = tokenizers_result['attention_mask']

    # Tokenize the labels (true tokens)
    tokenized_labels = tokenizers.sequence(
        labels,
        return_tensors="pt",
        add_special_tokens=False,  # No special tokens needed for single characters
        padding=False  # No padding needed for single tokens
    )['input_ids']

    return input_ids.to(device), sequence_ids.to(device), tokenized_labels.squeeze(-1).to(device), masked_indices.to(device)

In [4]:
from transformer_lens import HookedESM3,SupportedESM3Config
config = SupportedESM3Config(
    use_attn_result=True,
    use_split_qkv_input=True,
    use_hook_mlp_in=False,
    use_attn_in=True,
    esm3_output_type="all",
    esm3_use_torch_layer_norm=True,
    esm3_use_torch_attention_calc=True,
    esm3_use_org_rotary=True
)

esm3_hooked = HookedESM3.from_pretrained(esm_cfg=config, device=device)

If using ESM3 for interpretability research, keep in mind that ESM3 has some significant architectural differences to Language transformers like GPT.


Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

/home/galkesten/miniconda3/envs/transformer_lens/lib/python3.10/site-packages/esm/pretrained.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(


Moving model to device:  cuda
Loaded pretrained model esm3_sm_open_v1 into HookedESM3


In [5]:
from esm.pretrained import (
    ESM3_sm_open_v0,
)

esm3_original = ESM3_sm_open_v0(device).to(device)

In [6]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
file_path ="proteins_faithfulness.csv"
df = pd.read_csv(file_path).head(16)

In [7]:
proteins_list = []
for index,row in df.iterrows():
    proteins_list.append(row['protein'])

In [8]:
input_ids, sequence_ids, tokenized_labels, masked_indices = prepare_sequences(proteins_list, device)


In [9]:
torch.cuda.empty_cache()  # Clear GPU cache (optional, if using GPU)

In [10]:
import time

# Start the timer


# Perform the forward pass
with torch.no_grad():
    start_time = time.time()
    output1 = esm3_hooked.forward(
        sequence_tokens=input_ids,
        sequence_id=sequence_ids
    )
    
    # Stop the timer
    end_time = time.time()
    
    # Calculate the elapsed time
    inference_time = end_time - start_time
    print(f"Inference time: {inference_time:.6f} seconds")

del output1

Inference time: 4.913979 seconds


In [11]:

import time

# Start the timer
start_time = time.time()

# Perform the forward pass
output2 = esm3_original.forward(
        sequence_tokens=input_ids,
        sequence_id=sequence_ids)

# Stop the timer
end_time = time.time()

# Calculate the elapsed time
inference_time = end_time - start_time
print(f"Inference time: {inference_time:.6f} seconds")


Inference time: 0.214609 seconds


In [12]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))


True
1
NVIDIA L40S
